# Pipeline — fase 2: hidratação seletiva e rotulagem

Orquestra a **fase 2** do trabalho sobre os resultados finais da fase 1
(`notebooks/pipeline.ipynb`): para **todos os eventos de uma vez**, seleciona os tweets a
hidratar por cluster (D6), hidrata-os pela API do X usando o banco
(`data/database/hydrated.sqlite`) como cache e, em seguida, rotulará. Cada etapa tem uma
célula de inspeção.

Etapas: **Banco** (schema + eventos) → **M8** seleção por cluster → **Consolidação** (gravação
no banco + plano de custo) → **M9** hidratação de tweets → **M9** autores → **M7** coordenadas DRL
→ **M10** exportação para a visualização (`data/export/`) → **Status**.

**Idempotência.** Toda célula pode ser re-executada sem efeito colateral acumulado:
- estágios (`.run(...)`) usam o cache por existência de arquivo da fase 1 (`FORCE_FROM`);
- o banco só recebe *upserts* (tweets, autores, eventos) ou *replace por evento* (seleção);
- a hidratação busca **apenas o que falta** no banco — re-executar sem novidade não faz
  chamada alguma à API;
- a exportação (M10) sempre re-executa (custa < 1 s) e é determinística para o mesmo banco.

**Pré-requisitos por evento:** `graph_nodes.parquet` + `graph_edges.parquet` + `run_config.json`
em `data/processed/<evento>/`. O `retweets.parquet` (M1) é regenerado dos CSVs brutos se faltar.
**Custo:** a API é pay-per-use (US$ 0,005/tweet e US$ 0,010/autor **devolvidos**); a célula
de consolidação mostra quanto a hidratação vai custar antes de o M9 rodar.

In [1]:
import sys
from pathlib import Path

# Raiz do projeto: o notebook vive em notebooks/, mas pode rodar de notebooks/
# (Jupyter) ou da raiz do repositório (nbconvert/CI).
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))  # habilita `from modules.* import ...`

import json
import pandas as pd

from modules.database import Database
from modules.load import RetweetLoader
from modules.select_tweets import TopTweetSelector, load_final_graph
from modules.fetch_x_data import XClient, COST_PER_TWEET, COST_PER_USER
from modules.hydrate import TweetHydrator, UserHydrator, hydration_status
from modules.community import CommunityResult
from modules.layout import CommunityMap
from modules.export import WebExporter, write_index

## Configuração

A fase 2 roda sobre **todos** os eventos, porque a hidratação deduplica IDs entre eles e o
custo é global. Os parâmetros da seleção são os do D6 revisado (2026-09-10).

In [2]:
# ---- Eventos (subpastas de data/processed/) ----
EVENTOS = ["mobilizacao-0709", "roberto-jefferson", "eleicoes", "invasao-3-poderes"]

# ---- Parâmetros da seleção (D6, revisão 2026-09-10) ----
K = 100                   # top-K por cluster
K_SMALL = 20              # K para clusters com pouco peso interno
MIN_FRAC = 0.01           # cluster entra se tiver ≥ 1% dos nós (= min_frac da fase 1)
SMALL_WEIGHT_FRAC = 0.05  # ≤ 5% do peso TOTAL do grafo em arestas internas → K_SMALL

# ---- Hidratação ----
# Autores são um passo separado (D6). Ligado em 2026-09-15: a visualização (D15) precisa de
# nome, @handle e avatar. ≈ US$ 0,010 por autor devolvido.
HIDRATAR_AUTORES = True

# ---- Exportação para a visualização (M10) ----
EXPORT_DIR = PROJECT_ROOT / "data" / "export"

# ---- Cache por estágio ----
# FORCE_FROM = None -> usa cache onde houver.  FORCE_FROM = 8 -> recomputa M8 (e posteriores).
# No M9, "forçar" significa re-tentar os IDs que a API não devolveu (não custa nada).
# M7 (DRL, ~1–2 min/evento) só recalcula com FORCE_FROM ≤ 7 ou se faltar drl_layout.*.
FORCE_FROM = None

def _force(n: int) -> bool:
    return FORCE_FROM is not None and n >= FORCE_FROM

DB_PATH = PROJECT_ROOT / "data" / "database" / "hydrated.sqlite"
RAW = {ev: PROJECT_ROOT / "data" / "raw" / ev for ev in EVENTOS}
PROCESSED = {ev: PROJECT_ROOT / "data" / "processed" / ev for ev in EVENTOS}

for ev in EVENTOS:
    for f in ("graph_nodes.parquet", "graph_edges.parquet", "run_config.json"):
        assert (PROCESSED[ev] / f).exists(), f"{ev}: falta {f} — rode notebooks/pipeline.ipynb para esse evento"
    cfg = json.loads((PROCESSED[ev] / "run_config.json").read_text())
    print(f"{ev:20s} N={cfg['min_user_retweets']:<2} τ={cfg['tau']} min_frac={cfg['min_frac']} "
          f"gerado em {cfg['generated_at']}")
print(f"\nBanco:   {DB_PATH}")
print(f"Seleção: K={K}, K_SMALL={K_SMALL}, MIN_FRAC={MIN_FRAC}, SMALL_WEIGHT_FRAC={SMALL_WEIGHT_FRAC}")
print(f"Autores: {'hidratar' if HIDRATAR_AUTORES else 'não hidratar (HIDRATAR_AUTORES=False)'}")
print(f"Export:  {EXPORT_DIR}")

mobilizacao-0709     N=3  τ=0.1 min_frac=0.01 gerado em 2026-06-28T04:17:37
roberto-jefferson    N=8  τ=0.1 min_frac=0.01 gerado em 2026-06-28T17:49:17
eleicoes             N=5  τ=0.1 min_frac=0.01 gerado em 2026-06-28T18:03:26
invasao-3-poderes    N=7  τ=0.1 min_frac=0.01 gerado em 2026-06-28T04:07:16

Banco:   /home/vinicius/tcc/data/database/hydrated.sqlite
Seleção: K=100, K_SMALL=20, MIN_FRAC=0.01, SMALL_WEIGHT_FRAC=0.05
Autores: hidratar
Export:  /home/vinicius/tcc/data/export


## Banco — conexão, schema e eventos

`Database` garante o schema de forma idempotente (`CREATE TABLE IF NOT EXISTS`; ver
`data/database/README.md`). A tabela `events` é re-semeada por *upsert* com o `slug` igual ao
nome da pasta em `data/processed/`.

In [3]:
db = Database(DB_PATH)

EVENTS_ROWS = [
    {"slug": "mobilizacao-0709",  "name": "Mobilização do 7 de setembro de 2022",
     "event_date": "2022-09-07", "notes": "raw: data/raw/mobilizacao-0709/ (0709_mobilizacao.csv)"},
    {"slug": "roberto-jefferson", "name": "Caso Roberto Jefferson",
     "event_date": "2022-10-23", "notes": "raw: data/raw/roberto-jefferson/ (2310_robertojefferson_*.csv)"},
    {"slug": "eleicoes",          "name": "Debate sobre democracia no dia do 2º turno",
     "event_date": "2022-10-30", "notes": "raw: data/raw/eleicoes/ (3010_democracia.csv)"},
    {"slug": "invasao-3-poderes", "name": "Ataques de 8 de janeiro de 2023",
     "event_date": "2023-01-08", "notes": "raw: data/raw/invasao-3-poderes/ (0801/0901_invasao-*.csv)"},
]
db.upsert_events(EVENTS_ROWS)
with db.conn:
    db.conn.execute("DELETE FROM events WHERE slug = 'democracia-3010'")   # slug antigo; no-op se já não existir

print("Eventos:")
for r in db.conn.execute("SELECT slug, event_date, name FROM events ORDER BY event_date"):
    print(f"  {r['slug']:20s} {r['event_date']}  {r['name']}")
print("\nLinhas por tabela:", db.table_counts())

Eventos:
  mobilizacao-0709     2022-09-07  Mobilização do 7 de setembro de 2022
  roberto-jefferson    2022-10-23  Caso Roberto Jefferson
  eleicoes             2022-10-30  Debate sobre democracia no dia do 2º turno
  invasao-3-poderes    2023-01-08  Ataques de 8 de janeiro de 2023

Linhas por tabela: {'author_classification': 0, 'community_membership': 0, 'event_top_tweets': 1340, 'events': 4, 'lookup_errors': 197, 'tweets': 683, 'users': 385}


## Módulo 8 — Seleção por cluster (D6)

Para cada evento: comunidades com ≥ `MIN_FRAC` dos nós; ranking pelo nº de **membros do cluster**
que retuitaram (usuário×tweet distinto conta 1); `K` por cluster, ou `K_SMALL` quando o peso das
arestas internas do cluster é ≤ `SMALL_WEIGHT_FRAC` do peso **total** do grafo. Desempate
determinístico (`rt_graph` desc, `tweet_id` asc). Persiste `top_tweets.parquet` +
`top_tweets_stats.json` em `data/processed/<evento>/`.

O mesmo tweet pode aparecer em mais de um cluster — a deduplicação é feita na consolidação.

In [4]:
SELECOES, STATS = {}, {}
for ev in EVENTOS:
    print(f"\n== {ev}")
    nodes, edges = load_final_graph(PROCESSED[ev])
    retweets = RetweetLoader(RAW[ev]).run(out_dir=PROCESSED[ev])        # M1: regenera se faltar
    sel = TopTweetSelector(k=K, k_small=K_SMALL, min_frac=MIN_FRAC,
                           small_weight_frac=SMALL_WEIGHT_FRAC)
    SELECOES[ev] = sel.run(nodes, edges, retweets, out_dir=PROCESSED[ev], force=_force(8))
    STATS[ev] = s = sel.stats
    print(f"   {s['n_nodes']:,} nós | {len(s['clusters'])} clusters selecionados "
          f"(+{s['n_excluded_clusters']} abaixo de {MIN_FRAC:.0%}, {s['excluded_nodes_frac']:.1%} dos nós) | "
          f"{s['n_slots']} slots -> {s['n_unique_ids']} IDs únicos ({s['n_overlap']} repetidos entre clusters)")


== mobilizacao-0709


[cache] RetweetLoader: hit


[cache] TopTweetSelector: hit
   16,495 nós | 4 clusters selecionados (+6 abaixo de 1%, 0.8% dos nós) | 320 slots -> 224 IDs únicos (96 repetidos entre clusters)

== roberto-jefferson


[cache] RetweetLoader: hit


[cache] TopTweetSelector: hit
   27,042 nós | 3 clusters selecionados (+1 abaixo de 1%, 0.1% dos nós) | 300 slots -> 217 IDs únicos (83 repetidos entre clusters)

== eleicoes


[cache] RetweetLoader: hit
[cache] TopTweetSelector: hit
   9,742 nós | 5 clusters selecionados (+1 abaixo de 1%, 0.0% dos nós) | 420 slots -> 181 IDs únicos (239 repetidos entre clusters)

== invasao-3-poderes


[cache] RetweetLoader: hit


[cache] TopTweetSelector: hit
   33,305 nós | 3 clusters selecionados (+8 abaixo de 1%, 0.1% dos nós) | 300 slots -> 251 IDs únicos (49 repetidos entre clusters)


In [5]:
# Inspeção M8 — um cluster por linha, todos os eventos
rows = []
for ev in EVENTOS:
    for c, cs in STATS[ev]["clusters"].items():
        rows.append({"evento": ev, "cluster": int(c), "nós": cs["n_nodes"],
                     "% nós": 100 * cs["frac_nodes"], "% peso interno": 100 * cs["intra_weight_frac"],
                     "K": cs["k"], "candidatos": cs["n_candidates"], "selecionados": cs["n_selected"]})
tabela_clusters = pd.DataFrame(rows)
print(tabela_clusters.to_string(index=False, formatters={"% nós": "{:.1f}".format,
                                                          "% peso interno": "{:.2f}".format,
                                                          "nós": "{:,}".format,
                                                          "candidatos": "{:,}".format}))

print("\nTop-3 de cada cluster (invasao-3-poderes):")
print(SELECOES["invasao-3-poderes"].groupby("community").head(3).to_string(index=False))

           evento  cluster    nós % nós % peso interno   K candidatos  selecionados
 mobilizacao-0709        0  7,525  45.6          26.49 100      3,393           100
 mobilizacao-0709        1  5,251  31.8          19.96 100      2,436           100
 mobilizacao-0709        2  3,154  19.1          33.01 100      1,854           100
 mobilizacao-0709        3    439   2.7           1.29  20        402            20
roberto-jefferson        1 15,972  59.1          17.50 100     11,796           100
roberto-jefferson        0  6,386  23.6          34.56 100      3,376           100
roberto-jefferson        2  4,646  17.2          17.32 100      2,535           100
         eleicoes        1  3,017  31.0          18.92 100      1,972           100
         eleicoes        0  2,788  28.6           5.64 100      2,172           100
         eleicoes        2  2,062  21.2          11.54 100      1,477           100
         eleicoes        4  1,541  15.8           5.47 100      1,301       

## Consolidação — gravação no banco e plano de hidratação

Grava a seleção de cada evento em `event_top_tweets` (*replace* por evento: o que saiu do
top-K numa re-seleção some — só a **seleção**; nada em `tweets`/`users` é tocado) e deriva o
plano: IDs únicos entre clusters e eventos, quantos já estão no cache (`tweets`) e quantos
faltam, com o custo pay-per-use.

In [6]:
selected_at = pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds")
for ev in EVENTOS:
    n = db.replace_event_top_tweets(ev, SELECOES[ev], selected_at=selected_at)
    print(f"{ev:20s} {n:4d} linhas gravadas em event_top_tweets")

ids_por_evento = {ev: set(SELECOES[ev]["tweet_id"]) for ev in EVENTOS}
todos = set().union(*ids_por_evento.values())
n_slots = sum(len(SELECOES[ev]) for ev in EVENTOS)
n_por_evento = sum(len(s) for s in ids_por_evento.values())
cached = db.cached_tweet_ids()
a_hidratar = db.pending_tweet_ids()          # selecionados − cache − IDs que a API já negou

print(f"\nslots (cluster×rank): {n_slots}")
print(f"IDs únicos por evento: {n_por_evento}  |  únicos no total: {len(todos)}  "
      f"(repetidos entre eventos: {n_por_evento - len(todos)})")
print(f"já no cache: {len(todos & cached)}  |  já negados pela API: {len(todos & db.errored_ids('tweet'))}  |  "
      f"a hidratar: {len(a_hidratar)}  ≈ US$ {len(a_hidratar) * COST_PER_TWEET:.2f} em tweets")
print(f"autores: estimativa ≈ US$ {0.80 * len(a_hidratar) * COST_PER_USER:.2f} "
      f"(razão observada de 0,80 autor/tweet)")

mobilizacao-0709      320 linhas gravadas em event_top_tweets
roberto-jefferson     300 linhas gravadas em event_top_tweets
eleicoes              420 linhas gravadas em event_top_tweets
invasao-3-poderes     300 linhas gravadas em event_top_tweets

slots (cluster×rank): 1340
IDs únicos por evento: 873  |  únicos no total: 873  (repetidos entre eventos: 0)
já no cache: 676  |  já negados pela API: 197  |  a hidratar: 0  ≈ US$ 0.00 em tweets
autores: estimativa ≈ US$ 0.00 (razão observada de 0,80 autor/tweet)


## Módulo 9 — Hidratação: tweets

`TweetHydrator` (em `modules/hydrate.py`) pede à API só os IDs selecionados que **não** estão
em `tweets` nem em `lookup_errors`, em lotes de 100 e **sem expansions**; cada lote é gravado
antes do próximo (o banco é o checkpoint). Os IDs que a API não devolve ficam em
`lookup_errors` com o motivo dado por ela — `Not Found Error` (tweet removido) ou
`Authorization Error` (conta suspensa/protegida) — e não são re-pedidos, a menos que se force
(`FORCE_FROM ≤ 9`). Só recursos devolvidos são cobrados.

In [7]:
client = XClient()                                       # bearer token do .env
relatorio_tweets = TweetHydrator(db, client, retry_errors=_force(9)).run(out_dir=DB_PATH.parent)
print(relatorio_tweets.summary())

[cache] TweetHydrator: frio
[M9] tweet: 0 pendentes em 0 lote(s)
0 tweets pendentes → 0 devolvidos, 0 não devolvidos (nenhum) · US$ 0.00


In [8]:
# Inspeção M9 — atrição por cluster (compromisso do D6) e motivos dados pela API
status = hydration_status(db)
print(status.to_string(index=False, formatters={"atricao": "{:.1%}".format}))

uniq = db.conn.execute("SELECT COUNT(DISTINCT tweet_id) FROM event_top_tweets").fetchone()[0]
hid = db.conn.execute("SELECT COUNT(*) FROM tweets WHERE tweet_id IN (SELECT tweet_id FROM event_top_tweets)").fetchone()[0]
neg = db.conn.execute("SELECT COUNT(*) FROM lookup_errors WHERE resource_type = 'tweet' "
                      "AND resource_id IN (SELECT tweet_id FROM event_top_tweets)").fetchone()[0]
print(f"\nIDs únicos selecionados: {uniq} | hidratados: {hid} | não devolvidos: {neg} | "
      f"pendentes: {uniq - hid - neg} | atrição global: {(uniq - hid) / uniq:.1%}")
print("\nMotivos registrados pela API:")
print(pd.read_sql("SELECT title, COUNT(*) AS n FROM lookup_errors WHERE resource_type = 'tweet' "
                  "GROUP BY title ORDER BY n DESC", db.conn).to_string(index=False))

print("\nAmostra — top-3 de cada cluster com texto (invasao-3-poderes):")
print(pd.read_sql("""
    SELECT e.community AS cluster, e.rank, e.rt_cluster, t.lang,
           COALESCE(substr(replace(t.text, char(10), ' '), 1, 90), '— não devolvido —') AS texto
    FROM event_top_tweets e LEFT JOIN tweets t ON t.tweet_id = e.tweet_id
    WHERE e.event_slug = 'invasao-3-poderes' AND e.rank <= 3
    ORDER BY e.community, e.rank""", db.conn).to_string(index=False))

           evento  cluster  selecionados  hidratados  nao_encontrados  nao_autorizados  outros_erros  pendentes atricao
         eleicoes        0           100          76               18                6             0          0   24.0%
         eleicoes        1           100          83               14                3             0          0   17.0%
         eleicoes        2           100          77               16                7             0          0   23.0%
         eleicoes        3            20          18                1                1             0          0   10.0%
         eleicoes        4           100          81               14                5             0          0   19.0%
invasao-3-poderes        0           100          84               11                5             0          0   16.0%
invasao-3-poderes        1           100          72               16               12             0          0   28.0%
invasao-3-poderes        2           100

## Amostra para leitura — top-3 de cada cluster

Os três tweets mais retuitados **pelos membros** de cada cluster, com texto, para a análise
subjetiva começar. `pureza` = `rt_cluster / rt_graph` (fração dos retweets vindos de nós do
grafo que são deste cluster: 1,00 = só este cluster retuitou). Se algum dos três não foi
devolvido pela API, o próximo do ranking entra e a nota do cluster avisa — a ausência no topo
também é dado.

In [9]:
TOP = 3
amostra = pd.read_sql("""
    SELECT e.event_slug AS evento, e.community AS cluster, e.rank, e.rt_cluster, e.rt_graph, e.k,
           t.text, le.title AS erro
    FROM event_top_tweets e
    LEFT JOIN tweets t         ON t.tweet_id = e.tweet_id
    LEFT JOIN lookup_errors le ON le.resource_type = 'tweet' AND le.resource_id = e.tweet_id
    ORDER BY e.event_slug, e.community, e.rank""", db.conn)

for ev in EVENTOS:
    print(f"\n{'=' * 100}\n{ev}")
    for c, cs in STATS[ev]["clusters"].items():                       # ordem de tamanho (desc)
        g = amostra[(amostra.evento == ev) & (amostra.cluster == int(c))]
        faltam = g[(g["rank"] <= TOP) & g.text.isna()]
        nota = ("  · não devolvidos no top-%d: " % TOP
                + ", ".join(f"rank {r} ({e})" for r, e in zip(faltam["rank"], faltam.erro))) if len(faltam) else ""
        print(f"\n-- c{c}  ({cs['frac_nodes']:.0%} dos nós, K={cs['k']}){nota}")
        for r in g[g.text.notna()].head(TOP).itertuples():
            print(f"  #{r.rank:<3} rt_cluster={r.rt_cluster:<5} pureza={r.rt_cluster / r.rt_graph:.2f}  "
                  f"{' '.join(r.text.split())}")


mobilizacao-0709

-- c0  (46% dos nós, K=100)  · não devolvidos no top-3: rank 1 (Not Found Error)
  #2   rt_cluster=1324  pureza=1.00  E apesar do 7 de setembro, haverá 8, haverá 9, 10.. e os nossos problemas reais ainda estarão aqui. A fome, a miséria e a falta de governo não vão passar da noite pro dia. Somos a maioria, não se curvem, dia 02 está chegando!
  #3   rt_cluster=1134  pureza=0.99  Bolsonaro não ganhou um voto hoje. Falou apenas para sua base de extrema-direita. O sequestro da data nacional, o crime eleitoral explícito, a ameaça de golpe e a fala machista de homem fraco só o isolam mais política e eleitoralmente. Bolsonaro perdeu de novo no 7 de Setembro.
  #4   rt_cluster=831   pureza=0.99  Se Freud estivesse vivo, em 7 de setembro de 2022, ele diria: Não há nenhum sinal maior da insegurança masculina do que ter que se declarar "imbroxável" aos gritos numa praça pública.

-- c1  (32% dos nós, K=100)  · não devolvidos no top-3: rank 2 (Authorization Error)
  #1   rt_clus

## Módulo 9 — Hidratação: autores

Passo separado por decisão (D6). Ligado desde 2026-09-15 porque a visualização (D15) renderiza
o card com nome, @handle e avatar. `UserHydrator` pede à API (`/2/users`, US$ 0,010 cada) só os
autores dos tweets em `tweets` que não estão em `users` nem em `lookup_errors`; contas suspensas
ou removidas ficam em `lookup_errors` e o card mostra o motivo. Com `HIDRATAR_AUTORES = False`
a célula só mostra a pendência e o custo.

In [10]:
pendentes_autores = db.pending_author_ids(retry_errors=_force(9))
if HIDRATAR_AUTORES:
    relatorio_autores = UserHydrator(db, client, retry_errors=_force(9)).run(out_dir=DB_PATH.parent)
    print(relatorio_autores.summary())
else:
    print(f"HIDRATAR_AUTORES = False — {len(pendentes_autores)} autores pendentes "
          f"(≈ US$ {len(pendentes_autores) * COST_PER_USER:.2f}); nenhuma chamada feita.")

# Inspeção — cobertura de autores dos tweets em cache
tot = db.conn.execute("SELECT COUNT(DISTINCT author_id) FROM tweets WHERE author_id IS NOT NULL").fetchone()[0]
hid = db.conn.execute("SELECT COUNT(DISTINCT t.author_id) FROM tweets t JOIN users u ON u.user_id = t.author_id").fetchone()[0]
neg = dict(db.conn.execute("SELECT title, COUNT(*) FROM lookup_errors WHERE resource_type = 'user' GROUP BY title").fetchall())
print(f"autores distintos dos tweets em cache: {tot} | em users: {hid} | não devolvidos: {neg or 'nenhum'}")

[cache] UserHydrator: frio
[M9] user: 0 pendentes em 0 lote(s)
0 usuários pendentes → 0 devolvidos, 0 não devolvidos (nenhum) · US$ 0.00
autores distintos dos tweets em cache: 379 | em users: 379 | não devolvidos: nenhum


## Módulo 7 (fase 1) — coordenadas DRL para a visualização

O mini-mapa do leitor de clusters (D15) usa as mesmas coordenadas DRL da figura da fase 1
(`CommunityMap`: top-K arestas por nó → componente gigante → DRL ponderado, semente fixa). Elas
deixam de ser intermediários e passam a ser **entregáveis** em `data/processed/<evento>/`
(`drl_layout.parquet` + `drl_layout.json`); a limpeza da fase 1 não as apaga mais. O grafo final é
carregado sem construir o igraph (`build_graph=False`), só quando há cálculo a fazer.

In [11]:
LAYOUTS = {}
for ev in EVENTOS:
    cfg = json.loads((PROCESSED[ev] / "run_config.json").read_text())
    cm = CommunityMap(top_k=cfg.get("layout_top_k", 12), seed=cfg.get("layout_seed", 42), min_frac=MIN_FRAC)
    precisa = _force(7) or not all((PROCESSED[ev] / f).exists() for f in CommunityMap.FILES)
    cr = CommunityResult.load(PROCESSED[ev], build_graph=False) if precisa else None   # W + membership, sem igraph
    print(f"\n== {ev}")
    LAYOUTS[ev] = cm.run(cr, out_dir=PROCESSED[ev], force=_force(7))
    lay = LAYOUTS[ev]
    print(f"   {len(lay.coords):,} nós plotados de {lay.n_nodes:,} | top_k={lay.top_k} seed={lay.seed}")
    del cr


== mobilizacao-0709
[cache] CommunityMap: hit
   16,491 nós plotados de 16,495 | top_k=12 seed=42

== roberto-jefferson
[cache] CommunityMap: hit
   27,042 nós plotados de 27,042 | top_k=12 seed=42

== eleicoes
[cache] CommunityMap: hit
   9,740 nós plotados de 9,742 | top_k=12 seed=42

== invasao-3-poderes
[cache] CommunityMap: hit
   33,303 nós plotados de 33,305 | top_k=12 seed=42


## Módulo 10 — Exportação para a visualização

`WebExporter` junta, por evento, seleção (M8), hidratação (M9) e coordenadas (M7) em JSON estático
em `data/export/<evento>/` (`event.json`, `tweets.json`, `layout.json`) e `write_index` monta o
`data/export/index.json`. É o que o app web lê, sem backend (spec §5.3). Sempre re-exporta; saída
determinística para o mesmo banco. Contrato dos arquivos:
`docs/superpowers/specs/2026-09-15-leitor-de-clusters-design.md`.

In [12]:
exporter = WebExporter(db)
for ev in EVENTOS:
    res = exporter.run(PROCESSED[ev], out_dir=EXPORT_DIR / ev)
    e = res["event"]
    hid = sum(c["n_hydrated"] for c in e["clusters"]); sel = sum(c["n_selected"] for c in e["clusters"])
    print(f"   {ev:20s} {len(res['tweets']):4d} slots ({hid}/{sel} hidratados) | {len(e['clusters'])} clusters | "
          f"layout {res['layout']['n_plotted']:,} nós")
idx = write_index(EXPORT_DIR, EVENTOS)
print(f"\n{idx.relative_to(PROJECT_ROOT)}")
for p in sorted(EXPORT_DIR.rglob("*.json")):
    print(f"  {str(p.relative_to(EXPORT_DIR)):40s} {p.stat().st_size / 1024:8.1f} KB")

[cache] WebExporter: frio


   mobilizacao-0709      320 slots (229/320 hidratados) | 4 clusters | layout 16,491 nós
[cache] WebExporter: frio


   roberto-jefferson     300 slots (213/300 hidratados) | 3 clusters | layout 27,042 nós
[cache] WebExporter: frio


   eleicoes              420 slots (335/420 hidratados) | 5 clusters | layout 9,740 nós
[cache] WebExporter: frio


   invasao-3-poderes     300 slots (245/300 hidratados) | 3 clusters | layout 33,303 nós

data/export/index.json
  eleicoes/event.json                           2.4 KB
  eleicoes/layout.json                        141.3 KB
  eleicoes/tweets.json                        719.8 KB
  index.json                                    8.7 KB
  invasao-3-poderes/event.json                  1.7 KB
  invasao-3-poderes/layout.json               506.4 KB
  invasao-3-poderes/tweets.json               509.6 KB
  mobilizacao-0709/event.json                   2.0 KB
  mobilizacao-0709/layout.json                253.5 KB
  mobilizacao-0709/tweets.json                491.3 KB
  roberto-jefferson/event.json                  1.7 KB
  roberto-jefferson/layout.json               413.2 KB
  roberto-jefferson/tweets.json               438.7 KB


## Status — banco e artefatos

In [13]:
print("Linhas por tabela:")
for t, n in db.table_counts().items():
    print(f"  {t:24s} {n:>6,}")
print("\nArtefatos da fase 2 por evento:")
for ev in EVENTOS:
    for f in ("retweets.parquet", "top_tweets.parquet", "top_tweets_stats.json",
              "drl_layout.parquet", "drl_layout.json"):
        p = PROCESSED[ev] / f
        print(f"  {ev:20s} {f:24s} {p.stat().st_size / 1024:>9,.1f} KB" if p.exists() else f"  {ev:20s} {f:24s} AUSENTE")
db.close()

Linhas por tabela:
  author_classification         0
  community_membership          0
  event_top_tweets          1,340
  events                        4
  lookup_errors               197
  tweets                      683
  users                       385

Artefatos da fase 2 por evento:
  mobilizacao-0709     retweets.parquet           3,496.2 KB
  mobilizacao-0709     top_tweets.parquet            12.0 KB
  mobilizacao-0709     top_tweets_stats.json          1.1 KB
  mobilizacao-0709     drl_layout.parquet           264.5 KB
  mobilizacao-0709     drl_layout.json                0.1 KB
  roberto-jefferson    retweets.parquet          11,739.5 KB
  roberto-jefferson    top_tweets.parquet            12.3 KB
  roberto-jefferson    top_tweets_stats.json          0.9 KB
  roberto-jefferson    drl_layout.parquet           428.6 KB
  roberto-jefferson    drl_layout.json                0.1 KB
  eleicoes             retweets.parquet           3,845.3 KB
  eleicoes             top_tweets.parqu